# ImageGrains 2.0 — Final 4-Model Headline Benchmark (Hugging Face ONNX vs PyTorch vs Ground Truth)

Self-contained benchmark comparing all **4 shipped ONNX models** on [`inYourOwnBrowser/imagegrains-2.0-onnx`](https://huggingface.co/inYourOwnBrowser/imagegrains-2.0-onnx/tree/main) (pinned revision `e79f7b348f6fdc0725b41270a3e22f5777984f26`) against **upstream PyTorch (`cellpose` 3.1.1.1 & 4.1.x)** and **Ground Truth (`IG2_data.zip`, 40 `official_split == 'test'` images)**:

| Model Key | Shipped HF File | Precision / Format | Size | Expected SHA-256 (LFS oid) |
| --- | --- | --- | --- | --- |
| `unet_fp32` | `ig2_unet_fp32.onnx` | U-Net FP32 (opset 17, `ReduceMean` style) | 25.26 MiB | `5e692c55…cd62053c` |
| `cpsam_fp16` | `ig2_cpsam_fp16.onnx` | Cellpose-SAM ViT-L FP16 (opset 18) | 588.76 MiB | `944a114a…3b6230d` |
| `cpsam_int8` | `ig2_cpsam_int8_wo_fp32scale.onnx` | Cellpose-SAM INT8 weight-only (`fp32` scale + `Cast`) | 301.62 MiB | `2cda0be5…8bab6f43` |
| `cpsam_fp4` | `ig2_cpsam_fp4_e2m1_lut.onnx` | Cellpose-SAM 4-bit FP4 E2M1 LUT | 174.80 MiB | `d49d8f07…44320c99` |

### What this notebook tests
1. **Integrity & Download Footprint:** Downloads directly from Hugging Face (with Drive cache), verifies exact byte sizes and SHA-256 hashes, and measures raw / gzip / brotli sizes.
2. **Browser Runtime (`onnxruntime-web@1.31.1` WASM EP):** Load time, peak RSS memory, per-tile latency (mean / median / p95), plus Python ORT CPU & CUDA comparison.
3. **23-Tile Numerical Parity vs PyTorch FP32 & FP16:** Max/mean abs error, RMSE/NRMSE, flow cosine similarity & angular error, `cellprob` sign agreement, non-finite count, and explicit all-zero `256×256` tile check.
4. **Full 40-Image Held-Out Evaluation (Shipped TS Pipeline):** Runs `runUnetCellposeInference` / `runCpsamCellposeInference` (`dist/modules/onnxInference.js`) + `flowsToMasksBrowser` (`maxGrainAreaFraction = 0.40`), alongside upstream PyTorch (`pt_unet`, `pt_cpsam`).
5. **Headline Statistical & Error Suite:** `PQ`, `SQ`, `RQ (F1@0.5)`, `AP@0.5`, `AP@0.75`, `D16/D50/D84` APE, signed bias, RMSE/MAE, Pearson $r$, Spearman $\rho$, $\log(D_{50})$ $R^2$ — **vs both Ground Truth and PyTorch**, 10,000-draw paired bootstrap CIs + sign tests, per-site breakdown, worst-case outliers, and decision charts.

**Recommended Colab runtime:** L4 or A100 GPU with **High-RAM**.


In [ ]:
# CONFIG — Final 4-Model Hugging Face Headline Benchmark
REPO_URL = "https://github.com/LukeAFullard/webGrainy.git"
REPO_BRANCH = "audit/quantization-fp8-fp4"

HF_REPO = "inYourOwnBrowser/imagegrains-2.0-onnx"
HF_REVISION = "e79f7b348f6fdc0725b41270a3e22f5777984f26"

HF_MODELS = {
    "unet_fp32": {
        "filename": "ig2_unet_fp32.onnx",
        "model_kind": "unet",
        "input_dtype": "float32",
        "expected_bytes": 26489665,
        "expected_sha256": "5e692c5593ac15e614b77602e1d4f8c2f84dd939f4386b1f7e5641dbcd62053c",
        "label": "U-Net FP32",
    },
    "cpsam_fp16": {
        "filename": "ig2_cpsam_fp16.onnx",
        "model_kind": "cpsam",
        "input_dtype": "float16",
        "expected_bytes": 617364567,
        "expected_sha256": "944a114a0cb1a3a215c11affefe7211caac6da2b74895c496d5a7b5a23b6230d",
        "label": "CP-SAM FP16",
    },
    "cpsam_int8": {
        "filename": "ig2_cpsam_int8_wo_fp32scale.onnx",
        "model_kind": "cpsam",
        "input_dtype": "float16",
        "expected_bytes": 316272029,
        "expected_sha256": "2cda0be5b81f1d0c0ac36998beb9f741e11833aacd089c4f5cee92488bab6f43",
        "label": "CP-SAM INT8-WO",
    },
    "cpsam_fp4": {
        "filename": "ig2_cpsam_fp4_e2m1_lut.onnx",
        "model_kind": "cpsam",
        "input_dtype": "float16",
        "expected_bytes": 183292650,
        "expected_sha256": "d49d8f0779af3956b45b506719702bc3f9b7e0044a2d2a0a6842751544320c99",
        "label": "CP-SAM FP4-LUT",
    },
}

EVAL_MAX_IMAGES = 40          # 40 = full IG2 official_split == 'test'
WASM_NUM_THREADS = 4          # Thread count for onnxruntime-web WASM
TS_EVAL_BACKEND = "strict_auto"  # "strict_auto" (CPU bridge only if max|WASM-CPU|<0.05), "wasm", or "cpu"
SEED = 42
N_BOOT = 2000
PAIRED_BOOT = 10000

LOCAL_WORK_DIR = "/content/final_4model_bench"
DRIVE_OUT_DIR = "/content/drive/MyDrive/imagegrains_final_4model_benchmark"
DRIVE_QUANT_MODELS_DIR = "/content/drive/MyDrive/imagegrains_onnx_quant/models"
DRIVE_DATA_DIR = "/content/drive/MyDrive/imagegrains_datasets"
SKIP_CACHED_ON_DRIVE = True


In [ ]:
# Cell 1: Mount Drive, clone/build webGrainy, setup uv venvs (cp3 & cp4), download & SHA-256 verify HF models + IG2 data
import csv, gzip, hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from collections import Counter
import brotli
import psutil
from tqdm.auto import tqdm

os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
WEIGHTS_DIR = os.path.join(LOCAL_WORK_DIR, "weights")
DATA_DIR = os.path.join(LOCAL_WORK_DIR, "IG2_data")
MODELS_DIR = os.path.join(LOCAL_WORK_DIR, "models")
TILES_DIR = os.path.join(LOCAL_WORK_DIR, "tiles")
PT_CACHE_DIR = os.path.join(LOCAL_WORK_DIR, "pt_cache")
OUT_DIR = os.path.join(LOCAL_WORK_DIR, "outputs")
for d in (WEIGHTS_DIR, DATA_DIR, MODELS_DIR, TILES_DIR, PT_CACHE_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.exists("/content/drive/MyDrive")
except Exception as e:
    print("Drive mount skipped:", e)

if DRIVE_MOUNTED:
    for sub in ("", "models", "pt_cache", "cache", "plots"):
        os.makedirs(os.path.join(DRIVE_OUT_DIR, sub) if sub else DRIVE_OUT_DIR, exist_ok=True)

print(f"RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB | Free Disk: {shutil.disk_usage('/content').free / (1024**3):.1f} GB | Drive: {DRIVE_MOUNTED}")

def ensure_node22():
    try:
        v = subprocess.check_output(["node", "-v"], text=True).strip()
        if int(v.lstrip("v").split(".")[0]) >= 22:
            print("Node.js:", v)
            return
    except Exception:
        pass
    print("Installing Node.js 22...")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -q nodejs", shell=True, check=True)
    print("Node.js:", subprocess.check_output(["node", "-v"], text=True).strip())

ensure_node22()

REPO_DIR = "/content/webGrainy"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
chk = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", f"origin/{REPO_BRANCH}"], capture_output=True)
ref = f"origin/{REPO_BRANCH}" if chk.returncode == 0 else "origin/main"
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", "final_bench", ref], check=True)
print("Repo checked out:", ref, subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())

subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=REPO_DIR, check=True)
subprocess.run(["npm", "run", "build"], cwd=REPO_DIR, check=True)
subprocess.run(["node", os.path.join(REPO_DIR, "audit/followup/ts_cellpose_runner.mjs"), "--selftest"], cwd=REPO_DIR, check=True)
subprocess.run(["node", os.path.join(REPO_DIR, "audit/followup/ts_postprocess_runner.mjs"), "--selftest"], cwd=REPO_DIR, check=True)

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
ort_pkg = "onnxruntime-gpu>=1.20" if has_gpu else "onnxruntime>=1.20"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "onnx>=1.17", ort_pkg, "scikit-image", "scipy", "pandas", "matplotlib", "brotli", "ml_dtypes", "uv", "opencv-python-headless",
], check=True)

import onnx

def setup_uv_venv(venv_dir: str, pkgs: list[str]):
    py = os.path.join(venv_dir, "bin", "python")
    if os.path.exists(py):
        return py
    subprocess.run(["uv", "venv", venv_dir, "--python", sys.executable], check=True)
    torch_idx = ["--index-url", "https://download.pytorch.org/whl/cu124"] if has_gpu else []
    subprocess.run(["uv", "pip", "install", "--python", py, "torch", "torchvision", *torch_idx], check=True)
    subprocess.run(["uv", "pip", "install", "--python", py, *pkgs], check=True)
    return py

VENV_CP3_PY = setup_uv_venv("/content/venv_cp3", ["cellpose==3.1.1.1", "numpy<2.0", "opencv-python-headless", "pillow", "tqdm"])
VENV_CP4_PY = setup_uv_venv("/content/venv_cp4", ["cellpose>=4.0.9,<4.2", "opencv-python-headless", "pillow", "tqdm"])

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(4 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def download_file(url: str, dst: str):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return
    print("Downloading:", url)
    tmp = dst + ".part"
    with urllib.request.urlopen(url) as resp, open(tmp, "wb") as out:
        total = int(resp.headers.get("Content-Length") or 0)
        with tqdm(total=total or None, unit="B", unit_scale=True, desc=os.path.basename(dst)) as pbar:
            while True:
                buf = resp.read(1024 * 1024)
                if not buf:
                    break
                out.write(buf)
                pbar.update(len(buf))
    os.replace(tmp, dst)

# Download Zenodo PyTorch checkpoints
download_file("https://zenodo.org/records/15728186/files/IG2_full_set.200525?download=1", os.path.join(WEIGHTS_DIR, "IG2_full_set.200525"))
download_file("https://zenodo.org/records/15728186/files/IG2_full_set_cp_SAM?download=1", os.path.join(WEIGHTS_DIR, "IG2_full_set_cp_SAM"))

# Download & extract IG2_data.zip (check Drive cache first)
MANIFEST_CSV = os.path.join(REPO_DIR, "audit/followup/heldout_manifest.csv")
with open(MANIFEST_CSV, newline="", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

if any(not os.path.exists(os.path.join(DATA_DIR, r["image_relpath"])) for r in manifest_rows):
    ig2_zip = os.path.join(LOCAL_WORK_DIR, "IG2_data.zip")
    drive_zip = os.path.join(DRIVE_DATA_DIR, "IG2_data.zip")
    if DRIVE_MOUNTED and os.path.exists(drive_zip):
        shutil.copy2(drive_zip, ig2_zip)
    else:
        download_file("https://zenodo.org/records/17866827/files/IG2_data.zip?download=1", ig2_zip)
        if DRIVE_MOUNTED:
            os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
            shutil.copy2(ig2_zip, drive_zip)
    with zipfile.ZipFile(ig2_zip, "r") as zf:
        zf.extractall(DATA_DIR)

test_rows = [r for r in manifest_rows if r["official_split"] == "test"][:EVAL_MAX_IMAGES]
for r in test_rows:
    r["image_path"] = os.path.join(DATA_DIR, r["image_relpath"])
    r["mask_path"] = os.path.join(DATA_DIR, r["mask_relpath"])
print(f"Held-out dataset ready: {len(test_rows)} test images across sites {sorted({r['site'] for r in test_rows})}.")

# Download + strictly verify all 4 shipped ONNX models from Hugging Face
model_meta = {}
print("\nVerifying the 4 pinned Hugging Face models:")
for mkey, spec in HF_MODELS.items():
    fname = spec["filename"]
    local_p = os.path.join(MODELS_DIR, fname)
    for ddir in (os.path.join(DRIVE_OUT_DIR, "models"), DRIVE_QUANT_MODELS_DIR):
        cand = os.path.join(ddir, fname)
        if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE and not os.path.exists(local_p) and os.path.exists(cand):
            if os.path.getsize(cand) == spec["expected_bytes"] and sha256_file(cand) == spec["expected_sha256"]:
                shutil.copy2(cand, local_p)
                print(f"  [cache hit] {fname}")
                break
    if not os.path.exists(local_p):
        url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{fname}"
        download_file(url, local_p)

    actual_bytes = os.path.getsize(local_p)
    actual_sha = sha256_file(local_p)
    assert actual_bytes == spec["expected_bytes"], f"{fname} size mismatch: {actual_bytes} != {spec['expected_bytes']}"
    assert actual_sha == spec["expected_sha256"], f"{fname} SHA-256 mismatch: {actual_sha} != {spec['expected_sha256']}"

    if DRIVE_MOUNTED:
        drive_p = os.path.join(DRIVE_OUT_DIR, "models", fname)
        if not os.path.exists(drive_p):
            shutil.copy2(local_p, drive_p)

    raw_bytes = open(local_p, "rb").read()
    gz_len = len(gzip.compress(raw_bytes, compresslevel=6))
    br_len = len(brotli.compress(raw_bytes, quality=6))
    proto = onnx.load(local_p, load_external_data=False)
    op_counts = dict(Counter(n.op_type for n in proto.graph.node))
    model_meta[mkey] = {
        **spec,
        "path": local_p,
        "size": {
            "bytes": actual_bytes,
            "mb": round(actual_bytes / (1024 * 1024), 2),
            "gzip_bytes": gz_len,
            "gzip_mb": round(gz_len / (1024 * 1024), 2),
            "brotli_bytes": br_len,
            "brotli_mb": round(br_len / (1024 * 1024), 2),
            "sha256": actual_sha,
        },
        "opset": int(proto.opset_import[0].version),
        "ir_version": int(proto.ir_version),
        "node_count": len(proto.graph.node),
        "initializer_count": len(proto.graph.initializer),
        "op_counts": op_counts,
    }
    del proto, raw_bytes
    s = model_meta[mkey]["size"]
    print(f"  OK {mkey:10s} | {s['mb']:6.2f} MiB (gz {s['gzip_mb']:6.2f}, br {s['brotli_mb']:6.2f}) | opset={model_meta[mkey]['opset']} | sha256={actual_sha[:12]}...")


In [ ]:
# Cell 2: Generate PyTorch references — 23-tile raw flows/cellprob (FP32 & FP16) + 40-image PyTorch masks (pt_unet, pt_cpsam)
import json, os, shutil, subprocess
import cv2
import numpy as np

def load_rgb(path: str) -> np.ndarray:
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is not None:
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    from PIL import Image
    return np.array(Image.open(path).convert("RGB"))

def norm_rgb_1_99(img_rgb: np.ndarray) -> np.ndarray:
    out = np.empty_like(img_rgb, dtype=np.float32)
    for c in range(3):
        ch = img_rgb[:, :, c].astype(np.float32)
        p1, p99 = np.percentile(ch, [1, 99])
        denom = p99 - p1
        out[:, :, c] = (ch - p1) / denom if denom > 1e-3 else 0.0
    return np.transpose(out, (2, 0, 1))

def norm_gray_1_99(img_rgb: np.ndarray) -> np.ndarray:
    gray = img_rgb.astype(np.float32).mean(axis=2)
    p1, p99 = np.percentile(gray, [1, 99])
    denom = p99 - p1
    g = (gray - p1) / denom if denom > 1e-3 else np.zeros_like(gray)
    out = np.zeros((2, g.shape[0], g.shape[1]), dtype=np.float32)
    out[0] = g
    return out

def make_tiles(model_kind: str):
    C = 3 if model_kind == "cpsam" else 2
    tiles = []
    for idx, r in enumerate(test_rows[:20]):
        rgb = load_rgb(r["image_path"])
        arr = norm_rgb_1_99(rgb) if model_kind == "cpsam" else norm_gray_1_99(rgb)
        _, H, W = arr.shape
        if H < 256 or W < 256:
            arr = np.pad(arr, ((0, 0), (0, max(0, 256 - H)), (0, max(0, 256 - W))), mode="constant")
            y0, x0 = 0, 0
        else:
            y0, x0 = (H - 256) // 2, (W - 256) // 2
        tiles.append((f"real_{idx:02d}_{r['uid'][:24]}", arr[:, y0:y0+256, x0:x0+256].astype(np.float32)))
    tiles.append(("all_zero", np.zeros((C, 256, 256), dtype=np.float32)))
    t_b = tiles[0][1].copy()
    t_b[:, :8, :] = 0; t_b[:, -8:, :] = 0; t_b[:, :, :8] = 0; t_b[:, :, -8:] = 0
    tiles.append(("zero_border_8px", t_b))
    rgb_small = cv2.resize(load_rgb(test_rows[1]["image_path"]), (80, 100), interpolation=cv2.INTER_AREA)
    arr_small = norm_rgb_1_99(rgb_small) if model_kind == "cpsam" else norm_gray_1_99(rgb_small)
    t_pad = np.zeros((C, 256, 256), dtype=np.float32)
    t_pad[:, :100, :80] = arr_small
    tiles.append(("padded_100x80", t_pad))
    return tiles

pt_worker_py = r'''
import argparse, json, os, time
import cv2
import numpy as np
import torch
from PIL import Image
from cellpose import models

def load_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is not None:
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return np.array(Image.open(path).convert("RGB"))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["unet", "cpsam"], required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--tiles-npy", required=True)
    ap.add_argument("--pt-fp32-out", required=True)
    ap.add_argument("--pt-fp16-out", default="")
    ap.add_argument("--jobs", required=True)
    ap.add_argument("--masks-dir", required=True)
    ap.add_argument("--timings-out", required=True)
    args = ap.parse_args()

    use_gpu = torch.cuda.is_available()
    device = torch.device("cuda" if use_gpu else "cpu")
    cp_model = models.CellposeModel(gpu=use_gpu, pretrained_model=args.weights)

    # 1. Run 40-image PyTorch eval FIRST on the untouched CellposeModel
    with open(args.jobs) as f:
        jobs = json.load(f)
    os.makedirs(args.masks_dir, exist_ok=True)
    timings = json.load(open(args.timings_out)) if os.path.exists(args.timings_out) else {}
    pending = [j for j in jobs if not os.path.exists(os.path.join(args.masks_dir, f"{j['uid']}.npy"))]
    for idx, j in enumerate(pending, 1):
        img_rgb = load_rgb(j["image_path"])
        t0 = time.perf_counter()
        if args.model == "unet":
            masks = cp_model.eval(
                img_rgb, channels=[0, 0], diameter=30.0,
                flow_threshold=0.4, cellprob_threshold=0.0, min_size=30, normalize=True,
            )[0].astype(np.int32)
        else:
            masks = cp_model.eval(
                img_rgb, diameter=None,
                flow_threshold=0.4, cellprob_threshold=0.0, min_size=30, tile_overlap=0.25, normalize=True,
            )[0].astype(np.int32)
        dt_ms = (time.perf_counter() - t0) * 1000.0
        np.save(os.path.join(args.masks_dir, f"{j['uid']}.npy"), masks)
        timings[j["uid"]] = dt_ms
        if idx % 10 == 0 or idx == len(pending):
            with open(args.timings_out, "w") as tf:
                json.dump(timings, tf)
            print(f"[{args.model}] PyTorch masks {idx}/{len(pending)} done", flush=True)
    if not os.path.exists(args.timings_out):
        with open(args.timings_out, "w") as tf:
            json.dump(timings, tf)

    # 2. Capture 23-tile PyTorch FP32 reference
    tiles = np.load(args.tiles_npy)
    if not os.path.exists(args.pt_fp32_out):
        net32 = cp_model.net.float().to(device).eval()
        if hasattr(net32, "dtype"):
            net32.dtype = torch.float32
        if hasattr(net32, "mkldnn"):
            net32.mkldnn = False
        refs32 = []
        with torch.no_grad():
            for t in tiles:
                out = net32(torch.from_numpy(t[None]).to(device))
                y = out[0] if isinstance(out, (tuple, list)) else out
                refs32.append(y.float().cpu().numpy()[0])
        np.save(args.pt_fp32_out, np.stack(refs32, axis=0).astype(np.float32))
        print(f"[{args.model}] Saved PyTorch FP32 23-tile reference.", flush=True)

    # 3. Capture 23-tile PyTorch FP16 reference on a fresh model instance (GPU only)
    if args.pt_fp16_out and not os.path.exists(args.pt_fp16_out):
        if not use_gpu:
            print(f"[{args.model}] Skipping PyTorch FP16 23-tile capture (CPU Half conv2d not supported).", flush=True)
        else:
            del cp_model
            torch.cuda.empty_cache()
            cp16 = models.CellposeModel(gpu=True, pretrained_model=args.weights)
            net16 = cp16.net
            if hasattr(net16, "dtype"):
                net16.dtype = torch.float16
            net16 = net16.to(dtype=torch.float16, device=device).eval()
            refs16 = []
            with torch.no_grad():
                for t in tiles:
                    out = net16(torch.from_numpy(t[None]).to(dtype=torch.float16, device=device))
                    y = out[0] if isinstance(out, (tuple, list)) else out
                    refs16.append(y.float().cpu().numpy()[0])
            np.save(args.pt_fp16_out, np.stack(refs16, axis=0).astype(np.float32))
            print(f"[{args.model}] Saved PyTorch FP16 23-tile reference.", flush=True)

if __name__ == "__main__":
    main()
'''
with open("/tmp/pt_ref_worker.py", "w") as f:
    f.write(pt_worker_py)

pt_jobs_path = os.path.join(LOCAL_WORK_DIR, "pt_eval_jobs.json")
with open(pt_jobs_path, "w") as f:
    json.dump([{"uid": r["uid"], "image_path": r["image_path"]} for r in test_rows], f)

tile_names = {}
pt_timings = {}
for mkind, py_bin, w_name in [
    ("unet", VENV_CP3_PY, "IG2_full_set.200525"),
    ("cpsam", VENV_CP4_PY, "IG2_full_set_cp_SAM"),
]:
    tlist = make_tiles(mkind)
    tile_names[mkind] = [x[0] for x in tlist]
    tiles_npy = os.path.join(TILES_DIR, f"{mkind}_tiles.npy")
    np.save(tiles_npy, np.stack([x[1] for x in tlist], axis=0))

    pt32_npy = os.path.join(TILES_DIR, f"{mkind}_pt_fp32_tiles.npy")
    pt16_npy = os.path.join(TILES_DIR, f"{mkind}_pt_fp16_tiles.npy") if mkind == "cpsam" else ""
    masks_dir = os.path.join(PT_CACHE_DIR, mkind)
    os.makedirs(masks_dir, exist_ok=True)
    timings_p = os.path.join(PT_CACHE_DIR, f"{mkind}_timings.json")

    if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE:
        drive_mdir = os.path.join(DRIVE_OUT_DIR, "pt_cache", mkind)
        if os.path.exists(drive_mdir):
            for fn in os.listdir(drive_mdir):
                dst_f = os.path.join(masks_dir, fn)
                if not os.path.exists(dst_f):
                    shutil.copy2(os.path.join(drive_mdir, fn), dst_f)
        for p in [pt32_npy, pt16_npy, timings_p]:
            if p and not os.path.exists(p):
                dp = os.path.join(DRIVE_OUT_DIR, "pt_cache", os.path.basename(p))
                if os.path.exists(dp):
                    shutil.copy2(dp, p)

    cmd = [
        py_bin, "/tmp/pt_ref_worker.py",
        "--model", mkind,
        "--weights", os.path.join(WEIGHTS_DIR, w_name),
        "--tiles-npy", tiles_npy,
        "--pt-fp32-out", pt32_npy,
        "--jobs", pt_jobs_path,
        "--masks-dir", masks_dir,
        "--timings-out", timings_p,
    ]
    if pt16_npy:
        cmd.extend(["--pt-fp16-out", pt16_npy])
    proc_r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if proc_r.stdout:
        print(proc_r.stdout)
    if proc_r.returncode != 0:
        raise RuntimeError(f"pt_ref_worker failed (exit {proc_r.returncode}):\n{proc_r.stdout[-3000:]}")
    pt_timings[mkind] = json.load(open(timings_p))

    if DRIVE_MOUNTED:
        drive_mdir = os.path.join(DRIVE_OUT_DIR, "pt_cache", mkind)
        os.makedirs(drive_mdir, exist_ok=True)
        for fn in os.listdir(masks_dir):
            dst_d = os.path.join(drive_mdir, fn)
            if not os.path.exists(dst_d):
                shutil.copy2(os.path.join(masks_dir, fn), dst_d)
        for p in [pt32_npy, pt16_npy, timings_p]:
            if p and os.path.exists(p):
                shutil.copy2(p, os.path.join(DRIVE_OUT_DIR, "pt_cache", os.path.basename(p)))


In [ ]:
# Cell 3: ORT-Web 1.31.1 WASM load, RSS, speed, & 23-tile numerical parity vs PyTorch FP32/FP16 (+ CPU & CUDA)
import json, os, shutil, subprocess, tempfile, time
import numpy as np
import onnxruntime as ort

wasm_bench_js = r'''
import fs from "node:fs";
import path from "node:path";
import { pathToFileURL } from "node:url";

const ortDist = path.join(process.cwd(), "node_modules/onnxruntime-web/dist");
const ort = await import(pathToFileURL(path.join(ortDist, "ort.min.mjs")).href);
const meta = JSON.parse(fs.readFileSync(process.argv[2], "utf8"));

ort.env.wasm.numThreads = Number(meta.numThreads || 4);
ort.env.wasm.simd = true;
ort.env.wasm.wasmPaths = ortDist + "/";

const rssMb = () => Math.round(process.memoryUsage().rss / (1024 * 1024) * 10) / 10;
let peakRss = rssMb();

const t0 = performance.now();
let session;
try {
  const bytes = new Uint8Array(fs.readFileSync(meta.modelPath));
  session = await ort.InferenceSession.create(bytes, { executionProviders: ["wasm"] });
  peakRss = Math.max(peakRss, rssMb());
} catch (err) {
  fs.writeFileSync(meta.resultJson, JSON.stringify({
    load_ok: false,
    load_error: String(err?.message || err),
    load_time_s: (performance.now() - t0) / 1000,
    peak_rss_mb: Math.max(peakRss, rssMb()),
  }));
  process.exit(0);
}
const loadTimeS = (performance.now() - t0) / 1000;

function f32ToF16Bits(f32) {
  const buf = new ArrayBuffer(4);
  const fv = new Float32Array(buf);
  const uv = new Uint32Array(buf);
  const out = new Uint16Array(f32.length);
  for (let i = 0; i < f32.length; i++) {
    fv[0] = f32[i];
    const b = uv[0];
    const s = (b >>> 16) & 0x8000;
    const e = (b >>> 23) & 0xff;
    const m = b & 0x7fffff;
    if (e === 0xff) { out[i] = s | 0x7c00 | (m ? 0x0200 : 0); continue; }
    if (e === 0) { out[i] = s; continue; }
    const ee = e - 127;
    if (ee > 15) { out[i] = s | 0x7c00; continue; }
    if (ee >= -14) {
      let he = ee + 15, hm = m >>> 13, rem = m & 0x1fff;
      if (rem > 0x1000 || (rem === 0x1000 && (hm & 1))) { hm++; if (hm === 0x400) { hm = 0; he++; } }
      out[i] = he >= 31 ? (s | 0x7c00) : (s | (he << 10) | hm);
    } else { out[i] = s; }
  }
  return out;
}

const raw = fs.readFileSync(meta.tilesPath);
const f32All = new Float32Array(raw.buffer, raw.byteOffset, raw.byteLength / 4);
const [N, C, H, W] = meta.tilesShape;
const tileElems = C * H * W;
const outTileElems = 3 * H * W;
const allOut = new Float32Array(N * outTileElems);
const timesMs = [];

try {
  for (let i = 0; i < N; i++) {
    const slice = f32All.subarray(i * tileElems, (i + 1) * tileElems);
    const tensor = meta.inputDtype === "float16"
      ? new ort.Tensor("float16", f32ToF16Bits(slice), [1, C, H, W])
      : new ort.Tensor("float32", slice.slice(), [1, C, H, W]);
    const st = performance.now();
    const res = await session.run({ image: tensor });
    const dt = performance.now() - st;
    if (i > 0) timesMs.push(dt);
    peakRss = Math.max(peakRss, rssMb());
    const outT = res.flows_cellprob;
    if (outT.type === "float16") {
      const u16 = outT.data;
      const dst = allOut.subarray(i * outTileElems, (i + 1) * outTileElems);
      for (let k = 0; k < outTileElems; k++) {
        const h = u16[k];
        const s = (h & 0x8000) << 16;
        const e = (h >>> 10) & 0x1f;
        const m = h & 0x03ff;
        let bits;
        if (e === 0) bits = m === 0 ? s : (() => { let mm = m, ee = 127 - 14; while (!(mm & 0x0400)) { mm <<= 1; ee--; } mm &= 0x03ff; return s | (ee << 23) | (mm << 13); })();
        else if (e === 31) bits = s | 0x7f800000 | (m << 13);
        else bits = s | ((e - 15 + 127) << 23) | (m << 13);
        const dv = new DataView(new ArrayBuffer(4));
        dv.setUint32(0, bits, true);
        dst[k] = dv.getFloat32(0, true);
      }
    } else {
      allOut.set(outT.data, i * outTileElems);
    }
  }
  fs.writeFileSync(meta.outPath, Buffer.from(allOut.buffer, allOut.byteOffset, allOut.byteLength));
  timesMs.sort((a, b) => a - b);
  const meanMs = timesMs.reduce((a, b) => a + b, 0) / Math.max(1, timesMs.length);
  const medMs = timesMs[Math.floor(timesMs.length / 2)] || meanMs;
  const p95Ms = timesMs[Math.min(timesMs.length - 1, Math.floor(timesMs.length * 0.95))] || meanMs;
  fs.writeFileSync(meta.resultJson, JSON.stringify({
    load_ok: true, run_ok: true, load_time_s: loadTimeS, peak_rss_mb: peakRss,
    avg_tile_ms: meanMs, median_tile_ms: medMs, p95_tile_ms: p95Ms, times_ms: timesMs,
  }));
} catch (err) {
  fs.writeFileSync(meta.resultJson, JSON.stringify({
    load_ok: true, run_ok: false, run_error: String(err?.message || err),
    load_time_s: loadTimeS, peak_rss_mb: peakRss,
  }));
}
'''
with open("/tmp/wasm_4model_bench.mjs", "w") as f:
    f.write(wasm_bench_js)

def bench_py_ort(model_path: str, tiles: np.ndarray, input_dtype: str, provider: str):
    if provider not in ort.get_available_providers():
        return None, None
    try:
        sess = ort.InferenceSession(model_path, providers=[provider])
        dt = np.float16 if input_dtype == "float16" else np.float32
        outs, times = [], []
        for i in range(len(tiles)):
            inp = tiles[i:i+1].astype(dt)
            t0 = time.perf_counter()
            y = sess.run(["flows_cellprob"], {"image": inp})[0][0].astype(np.float32)
            if i > 0:
                times.append((time.perf_counter() - t0) * 1000.0)
            outs.append(y)
        return float(np.mean(times)), np.stack(outs, axis=0)
    except Exception as e:
        print(f"  [py {provider}] error: {e}")
        return None, None

def compute_tile_numerics(pred: np.ndarray, ref: np.ndarray, source_label: str) -> dict:
    diff = np.abs(pred - ref)
    f_pred = pred[:20, :2].transpose(0, 2, 3, 1).reshape(-1, 2)
    f_ref = ref[:20, :2].transpose(0, 2, 3, 1).reshape(-1, 2)
    n_pred = np.linalg.norm(f_pred, axis=1)
    n_ref = np.linalg.norm(f_ref, axis=1)
    mask = (n_ref > 0.1) & (n_pred > 1e-6)
    if np.any(mask):
        cos = np.clip(np.sum(f_pred[mask] * f_ref[mask], axis=1) / (n_pred[mask] * n_ref[mask]), -1.0, 1.0)
        ang_deg = np.degrees(np.arccos(cos))
        cos_mean = float(np.mean(cos))
        ang_med = float(np.median(ang_deg))
    else:
        cos_mean, ang_med = float("nan"), float("nan")

    ref_std = float(np.std(ref)) + 1e-12
    nrmse = float(np.sqrt(np.mean((pred - ref) ** 2)) / ref_std)
    az_cp = pred[20, 2]
    az_ref = ref[20, 2]
    az_sign = float(100.0 * np.mean((az_cp < 0) == (az_ref < 0)))
    return {
        "source": source_label,
        "non_finite_count": int(np.sum(~np.isfinite(pred))),
        "nrmse_all": nrmse,
        "flow_cos_sim_mean": cos_mean,
        "flow_angle_err_median_deg": ang_med,
        "flowY_max_abs": float(np.nanmax(diff[:, 0])),
        "flowY_mean_abs": float(np.nanmean(diff[:, 0])),
        "flowX_max_abs": float(np.nanmax(diff[:, 1])),
        "flowX_mean_abs": float(np.nanmean(diff[:, 1])),
        "cellprob_max_abs": float(np.nanmax(diff[:, 2])),
        "cellprob_mean_abs": float(np.nanmean(diff[:, 2])),
        "cellprob_sign_agree_pct": float(100.0 * np.mean((pred[:, 2] < 0) == (ref[:, 2] < 0))),
        "all_zero_cellprob_mean": float(np.nanmean(az_cp)),
        "all_zero_cellprob_min": float(np.nanmin(az_cp)),
        "all_zero_cellprob_max": float(np.nanmax(az_cp)),
        "all_zero_sign_agree_pct": az_sign,
        "all_zero_negative_ok": bool(np.all(az_cp < 0) and az_sign >= 99.0),
    }

tile_results = {}
wasm_tile_outputs = {}

for mkey, spec in model_meta.items():
    mkind = spec["model_kind"]
    tiles = np.load(os.path.join(TILES_DIR, f"{mkind}_tiles.npy"))
    pt_fp32 = np.load(os.path.join(TILES_DIR, f"{mkind}_pt_fp32_tiles.npy"))
    pt_fp16_p = os.path.join(TILES_DIR, f"{mkind}_pt_fp16_tiles.npy")
    pt_fp16 = np.load(pt_fp16_p) if os.path.exists(pt_fp16_p) else None

    cache_json = os.path.join(OUT_DIR, f"tiles_{mkey}.json")
    cache_npy = os.path.join(TILES_DIR, f"{mkey}_wasm_out.npy")
    drive_cache = os.path.join(DRIVE_OUT_DIR, "cache", f"tiles_{mkey}.json")
    drive_npy = os.path.join(DRIVE_OUT_DIR, "cache", f"{mkey}_wasm_out.npy")
    if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE and os.path.exists(drive_cache) and os.path.exists(drive_npy):
        shutil.copy2(drive_cache, cache_json)
        shutil.copy2(drive_npy, cache_npy)
    if os.path.exists(cache_json) and os.path.exists(cache_npy):
        tile_results[mkey] = json.load(open(cache_json))
        wasm_tile_outputs[mkey] = np.load(cache_npy)
        print(f"[cached] {mkey}: az_cp={tile_results[mkey]['numerics_vs_pt_fp32']['all_zero_cellprob_mean']:.2f}")
        continue

    with tempfile.TemporaryDirectory() as td:
        tiles_p = os.path.join(td, "tiles.f32")
        out_p = os.path.join(td, "out.f32")
        res_p = os.path.join(td, "res.json")
        meta_p = os.path.join(td, "meta.json")
        np.ascontiguousarray(tiles, dtype="<f4").tofile(tiles_p)
        json.dump({
            "modelPath": spec["path"], "tilesPath": tiles_p, "tilesShape": list(tiles.shape),
            "inputDtype": spec["input_dtype"], "outPath": out_p, "resultJson": res_p, "numThreads": WASM_NUM_THREADS,
        }, open(meta_p, "w"))
        subprocess.run(["node", "/tmp/wasm_4model_bench.mjs", meta_p], cwd=REPO_DIR, check=True)
        wasm_res = json.load(open(res_p))
        wasm_out = np.fromfile(out_p, dtype="<f4").reshape(pt_fp32.shape) if wasm_res.get("run_ok") else None

    assert wasm_out is not None, f"{mkey} failed ORT-Web WASM: {wasm_res}"
    np.save(cache_npy, wasm_out)
    wasm_tile_outputs[mkey] = wasm_out

    py_cpu_ms, py_cpu_out = bench_py_ort(spec["path"], tiles, spec["input_dtype"], "CPUExecutionProvider")
    py_cuda_ms, py_cuda_out = bench_py_ort(spec["path"], tiles, spec["input_dtype"], "CUDAExecutionProvider")

    num_fp32 = compute_tile_numerics(wasm_out, pt_fp32, "wasm")
    num_fp16 = compute_tile_numerics(wasm_out, pt_fp16, "wasm") if pt_fp16 is not None else None
    cuda_az_mean = float(np.nanmean(py_cuda_out[20, 2])) if isinstance(py_cuda_out, np.ndarray) else None

    py_cpu_diff = np.abs(wasm_out - py_cpu_out) if isinstance(py_cpu_out, np.ndarray) else None
    py_cpu_max = float(np.max(py_cpu_diff)) if py_cpu_diff is not None else None
    py_cpu_mean = float(np.mean(py_cpu_diff)) if py_cpu_diff is not None else None

    rec = {
        "wasm": {k: v for k, v in wasm_res.items() if k != "times_ms"},
        "speed_ms": {
            "wasm_threads": WASM_NUM_THREADS,
            "wasm_mean_ms": wasm_res["avg_tile_ms"],
            "wasm_median_ms": wasm_res["median_tile_ms"],
            "wasm_p95_ms": wasm_res["p95_tile_ms"],
            "py_cpu_ms": py_cpu_ms,
            "py_cuda_ms": py_cuda_ms,
        },
        "numerics_vs_pt_fp32": num_fp32,
        "numerics_vs_pt_fp16": num_fp16,
        "cuda_all_zero_cellprob_mean": cuda_az_mean,
        "py_cpu_vs_wasm_max_abs": py_cpu_max,
        "py_cpu_vs_wasm_mean_abs": py_cpu_mean,
        "bridge_matches_wasm": bool(py_cpu_max is not None and py_cpu_max < 0.05),
    }
    tile_results[mkey] = rec
    json.dump(rec, open(cache_json, "w"), indent=2)
    if DRIVE_MOUNTED:
        shutil.copy2(cache_json, drive_cache)
        shutil.copy2(cache_npy, drive_npy)

    print(f"[{mkey:10s}] WASM={wasm_res['avg_tile_ms']:.0f}ms (med {wasm_res['median_tile_ms']:.0f}ms) | "
          f"RSS={wasm_res['peak_rss_mb']:.0f}MB | az_cp={num_fp32['all_zero_cellprob_mean']:.2f} "
          f"(CUDA az_cp={cuda_az_mean}) | pyCPU_diff(max={py_cpu_max}, mean={py_cpu_mean})")

for mkey in ("cpsam_int8", "cpsam_fp4"):
    num_onnx16 = compute_tile_numerics(wasm_tile_outputs[mkey], wasm_tile_outputs["cpsam_fp16"], "wasm")
    tile_results[mkey]["numerics_vs_onnx_fp16"] = num_onnx16
    print(f"[{mkey:10s} vs onnx_fp16] flowY_mean={num_onnx16['flowY_mean_abs']:.5f} cp_mean={num_onnx16['cellprob_mean_abs']:.5f} sign={num_onnx16['cellprob_sign_agree_pct']:.3f}%")


In [ ]:
# Cell 4: Run all 4 HF models through shipped TS pipeline (persistent Node session + ts_postprocess_runner) on 40 test images
import http.server, json, math, os, shutil, socketserver, subprocess, tempfile, threading, time
import numpy as np
import onnxruntime as ort
import pandas as pd
from skimage.measure import regionprops
from tqdm.auto import tqdm

TS_POSTPROCESS_RUNNER = os.path.join(REPO_DIR, "audit/followup/ts_postprocess_runner.mjs")

persistent_worker_js = r'''
import fs from "node:fs";
import readline from "node:readline";
import * as ort from "onnxruntime-web";
import { runUnetCellposeInference, runCpsamCellposeInference } from "./dist/modules/onnxInference.js";

function makeProxySession(bridgeUrl) {
  return {
    async run({ image }) {
      const body = Buffer.from(image.data.buffer, image.data.byteOffset, image.data.byteLength);
      const resp = await fetch(bridgeUrl, {
        method: "POST",
        headers: { "x-dtype": String(image.type), "x-dims": image.dims.join(",") },
        body,
      });
      if (!resp.ok) throw new Error(`Bridge HTTP ${resp.status}`);
      const ab = await resp.arrayBuffer();
      return { flows_cellprob: { data: new Float32Array(ab) } };
    },
  };
}

const cfg = JSON.parse(fs.readFileSync(process.argv[2], "utf8"));
let session;
if (cfg.bridgeUrl) {
  session = makeProxySession(cfg.bridgeUrl);
} else {
  ort.env.wasm.numThreads = Number(cfg.numThreads ?? 4);
  ort.env.wasm.simd = true;
  session = await ort.InferenceSession.create(fs.readFileSync(cfg.modelPath), { executionProviders: ["wasm"] });
}
console.log("READY");

const rl = readline.createInterface({ input: process.stdin, crlfDelay: Infinity });
for await (const line of rl) {
  if (!line.trim()) continue;
  const job = JSON.parse(line);
  if (job.quit) break;
  const nPix = job.width * job.height;
  const rgbaBuf = fs.readFileSync(job.rgbaPath);
  const rgba = new Uint8ClampedArray(rgbaBuf.buffer, rgbaBuf.byteOffset, rgbaBuf.byteLength);
  const t0 = performance.now();
  const out = cfg.modelKind === "unet"
    ? await runUnetCellposeInference(session, rgba, job.width, job.height)
    : await runCpsamCellposeInference(session, rgba, job.width, job.height);
  const elapsedMs = performance.now() - t0;
  const dp = new Float32Array(2 * nPix);
  dp.set(out.flowY, 0);
  dp.set(out.flowX, nPix);
  fs.writeFileSync(job.outDpPath, Buffer.from(dp.buffer, dp.byteOffset, dp.byteLength));
  fs.writeFileSync(job.outCpPath, Buffer.from(out.cellprob.buffer, out.cellprob.byteOffset, out.cellprob.byteLength));
  console.log(JSON.stringify({ ok: true, elapsedMs }));
}
'''
with open(os.path.join(LOCAL_WORK_DIR, "persistent_worker.mjs"), "w") as f:
    f.write(persistent_worker_js)

def load_gt_mask(path: str, hw: tuple[int, int]) -> np.ndarray:
    arr = np.load(path) if path.endswith(".npy") else cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if arr is None:
        from PIL import Image
        arr = np.array(Image.open(path))
    if arr.ndim == 3:
        arr = arr[:, :, 0] if np.array_equal(arr[:, :, 0], arr[:, :, 1]) else ((arr[:, :, 0].astype(np.int64) << 16) | (arr[:, :, 1].astype(np.int64) << 8) | arr[:, :, 2])
    if arr.shape[:2] != hw:
        from PIL import Image
        arr = np.array(Image.fromarray(arr.astype(np.int32), mode="I").resize((hw[1], hw[0]), Image.Resampling.NEAREST))
    arr_i = arr.astype(np.int64); arr_i[arr_i < 0] = 0
    uniq, inv = np.unique(arr_i, return_inverse=True)
    out = inv.reshape(hw).astype(np.int32)
    return out + 1 if (len(uniq) > 0 and uniq[0] != 0) else out

def compute_mask_grain_metrics(masks: np.ndarray, margin: int = 2) -> list[dict]:
    H, W = masks.shape
    grains = []
    for prop in regionprops(masks):
        area = float(prop.area)
        if area <= 0:
            continue
        minr, minc, maxr, maxc = prop.bbox
        if minr < margin or minc < margin or maxr > H - margin or maxc > W - margin:
            continue
        b_axis = float(prop.axis_minor_length) if hasattr(prop, "axis_minor_length") else float(prop.minor_axis_length)
        d_eq = math.sqrt(4.0 * area / math.pi)
        if b_axis > 0 and d_eq > 0:
            grains.append({"areaPx": area, "bAxisPx": b_axis, "dEqPx": d_eq})
    return grains

def area_weighted_percentiles(grains: list[dict], key: str, pcts=(16.0, 50.0, 84.0)) -> dict[str, float]:
    valid = [(float(g[key]), float(g["areaPx"])) for g in grains if not g.get("touchesBorder", g.get("isBorder", False)) and g.get(key, 0) > 0 and g.get("areaPx", 0) > 0]
    if not valid:
        return {f"D{int(p)}": float("nan") for p in pcts}
    valid.sort(key=lambda x: x[0])
    vals = np.array([v[0] for v in valid], dtype=np.float64)
    wts = np.array([v[1] for v in valid], dtype=np.float64)
    cdf = (np.cumsum(wts) - 0.5 * wts) / np.sum(wts) * 100.0
    return {f"D{int(p)}": float(np.interp(p, cdf, vals)) for p in pcts}

def compute_seg_metrics(pred: np.ndarray, ref: np.ndarray) -> dict[str, float]:
    pred_ids = np.unique(pred); pred_ids = pred_ids[pred_ids > 0]
    ref_ids = np.unique(ref); ref_ids = ref_ids[ref_ids > 0]
    np_, nr_ = len(pred_ids), len(ref_ids)
    if np_ == 0 and nr_ == 0:
        return {"pq": 1.0, "sq": 1.0, "rq": 1.0, "ap50": 1.0, "ap75": 1.0}
    if np_ == 0 or nr_ == 0:
        return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "ap50": 0.0, "ap75": 0.0}
    p_areas = {int(i): int((pred == i).sum()) for i in pred_ids}
    r_areas = {int(j): int((ref == j).sum()) for j in ref_ids}
    mask = (pred > 0) & (ref > 0)
    if not np.any(mask):
        return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "ap50": 0.0, "ap75": 0.0}
    pairs = pred[mask].astype(np.int64) * 1_000_000 + ref[mask].astype(np.int64)
    uniq_pairs, counts = np.unique(pairs, return_counts=True)
    cand50 = []
    for code, inter in zip(uniq_pairs, counts):
        pi, ri = int(code // 1_000_000), int(code % 1_000_000)
        iou = float(inter) / float(p_areas[pi] + r_areas[ri] - int(inter))
        if iou > 0.5:
            cand50.append((iou, pi, ri))
    cand50.sort(reverse=True)
    tp50, tp75, iou_sum50, mp, mr = 0, 0, 0.0, set(), set()
    for iou, pi, ri in cand50:
        if pi not in mp and ri not in mr:
            mp.add(pi); mr.add(ri)
            tp50 += 1; iou_sum50 += iou
            if iou >= 0.75:
                tp75 += 1
    fp50, fn50 = np_ - tp50, nr_ - tp50
    sq = float(iou_sum50 / tp50) if tp50 > 0 else 0.0
    rq = float(tp50 / (tp50 + 0.5 * fp50 + 0.5 * fn50))
    pq = float(iou_sum50 / (tp50 + 0.5 * fp50 + 0.5 * fn50))
    ap50 = float(tp50 / (tp50 + fp50 + fn50))
    ap75 = float(tp75 / (tp75 + (np_ - tp75) + (nr_ - tp75)))
    return {"pq": pq, "sq": sq, "rq": rq, "ap50": ap50, "ap75": ap75}

class _ThreadingHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True

def start_ort_bridge(sess: ort.InferenceSession):
    lock = threading.Lock()
    class Handler(http.server.BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"
        def log_message(self, *args): pass
        def do_POST(self):
            body = self.rfile.read(int(self.headers["Content-Length"]))
            dt = np.float16 if self.headers["x-dtype"] == "float16" else np.float32
            dims = [int(x) for x in self.headers["x-dims"].split(",")]
            arr = np.frombuffer(body, dtype=dt).reshape(dims)
            with lock:
                out = sess.run(["flows_cellprob"], {"image": arr})[0]
            payload = np.ascontiguousarray(out, dtype="<f4").tobytes()
            self.send_response(200); self.send_header("Content-Length", str(len(payload))); self.end_headers()
            self.wfile.write(payload)
    srv = _ThreadingHTTPServer(("127.0.0.1", 0), Handler)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    return srv, srv.server_address[1]

def run_ts_postprocess(dP: np.ndarray, cp: np.ndarray, guard: float = 0.40):
    H, W = cp.shape
    with tempfile.TemporaryDirectory() as td:
        dp_p, cp_p, meta_p = os.path.join(td, "dp.f32"), os.path.join(td, "cp.f32"), os.path.join(td, "meta.json")
        out_json, out_mask = os.path.join(td, "out.json"), os.path.join(td, "mask.i32")
        np.ascontiguousarray(dP, dtype="<f4").tofile(dp_p)
        np.ascontiguousarray(cp, dtype="<f4").tofile(cp_p)
        json.dump({
            "dPPath": dp_p, "cellprobPath": cp_p, "width": int(W), "height": int(H),
            "maxGrainAreaFraction": float(guard), "outputJsonPath": out_json, "outputMaskPath": out_mask,
        }, open(meta_p, "w"))
        subprocess.run(["node", TS_POSTPROCESS_RUNNER, "--input", meta_p], cwd=REPO_DIR, check=True)
        return np.fromfile(out_mask, dtype="<i4").reshape((H, W)), json.load(open(out_json))["grains"]

all_records = []
masks_root = os.path.join(LOCAL_WORK_DIR, "masks")
os.makedirs(masks_root, exist_ok=True)

for pkey, mkind in [("pt_unet", "unet"), ("pt_cpsam", "cpsam")]:
    rows_out = []
    for r in test_rows:
        uid = r["uid"]
        rgb = load_rgb(r["image_path"])
        H, W = rgb.shape[:2]
        gt_masks = load_gt_mask(r["mask_path"], (H, W))
        gt_grains = compute_mask_grain_metrics(gt_masks)
        gt_b = area_weighted_percentiles(gt_grains, "bAxisPx")
        pt_masks = np.load(os.path.join(PT_CACHE_DIR, mkind, f"{uid}.npy")).astype(np.int32)
        pt_grains = compute_mask_grain_metrics(pt_masks)
        pt_b = area_weighted_percentiles(pt_grains, "bAxisPx")
        pt_d = area_weighted_percentiles(pt_grains, "dEqPx")
        s_gt = compute_seg_metrics(pt_masks, gt_masks)
        rows_out.append({
            "uid": uid, "site": r["site"], "is_k1_overlap": r["is_k1_overlap"],
            "model_key": pkey, "model_kind": mkind, "backend": "pytorch",
            "infer_ms": pt_timings[mkind].get(uid, float("nan")),
            "n_gt": len(gt_grains), "n_pt": len(pt_grains), "n_pred": len(pt_grains),
            "grain_count_ratio_gt": len(pt_grains) / max(1, len(gt_grains)),
            "grain_count_ratio_pt": 1.0,
            "gt_b_D16": gt_b["D16"], "gt_b_D50": gt_b["D50"], "gt_b_D84": gt_b["D84"],
            "pt_b_D16": pt_b["D16"], "pt_b_D50": pt_b["D50"], "pt_b_D84": pt_b["D84"],
            "pred_b_D16": pt_b["D16"], "pred_b_D50": pt_b["D50"], "pred_b_D84": pt_b["D84"],
            "pred_d_D50": pt_d["D50"],
            "pq_vs_gt": s_gt["pq"], "sq_vs_gt": s_gt["sq"], "rq_vs_gt": s_gt["rq"], "ap50_vs_gt": s_gt["ap50"], "ap75_vs_gt": s_gt["ap75"],
            "pq_vs_pt": 1.0, "sq_vs_pt": 1.0, "rq_vs_pt": 1.0, "ap50_vs_pt": 1.0, "ap75_vs_pt": 1.0,
            "pq_vs_fp16": float("nan"), "ap50_vs_fp16": float("nan"),
        })
    all_records.extend(rows_out)

eval_order = ["unet_fp32", "cpsam_fp16", "cpsam_int8", "cpsam_fp4"]
fp16_masks = {}

for mkey in eval_order:
    spec = model_meta[mkey]
    mkind = spec["model_kind"]
    t_rec = tile_results[mkey]
    use_bridge = (TS_EVAL_BACKEND == "cpu") or (TS_EVAL_BACKEND == "strict_auto" and t_rec.get("bridge_matches_wasm", False))
    backend_label = "ort_cpu_bridge(wasm_verified)" if use_bridge else "ort_web_wasm"

    var_csv = os.path.join(OUT_DIR, f"eval_{mkey}.csv")
    drive_csv = os.path.join(DRIVE_OUT_DIR, "cache", f"eval_{mkey}.csv")
    if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE and os.path.exists(drive_csv) and not os.path.exists(var_csv):
        shutil.copy2(drive_csv, var_csv)

    existing = {}
    if os.path.exists(var_csv):
        for _, row in pd.read_csv(var_csv).iterrows():
            existing[row["uid"]] = row.to_dict()

    mdir = os.path.join(masks_root, mkey)
    os.makedirs(mdir, exist_ok=True)
    need_worker = any((r["uid"] not in existing) or (mkey == "cpsam_fp16" and not os.path.exists(os.path.join(mdir, f"{r['uid']}.npy"))) for r in test_rows)

    srv, proc = None, None
    if need_worker:
        bridge_url = None
        if use_bridge:
            sess = ort.InferenceSession(spec["path"], providers=["CPUExecutionProvider"])
            srv, port = start_ort_bridge(sess)
            bridge_url = f"http://127.0.0.1:{port}/{mkey}"
        wcfg = os.path.join(LOCAL_WORK_DIR, f"wcfg_{mkey}.json")
        json.dump({
            "modelKind": mkind, "modelPath": spec["path"], "bridgeUrl": bridge_url, "numThreads": WASM_NUM_THREADS,
        }, open(wcfg, "w"))
        proc = subprocess.Popen(
            ["node", os.path.join(LOCAL_WORK_DIR, "persistent_worker.mjs"), wcfg],
            cwd=REPO_DIR, stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1,
        )
        ready_line = proc.stdout.readline().strip()
        assert ready_line == "READY", f"Worker failed to start for {mkey}: {ready_line}"

    var_rows = []
    try:
        for r in tqdm(test_rows, desc=f"{mkey} [{backend_label}]"):
            uid = r["uid"]
            m_path = os.path.join(mdir, f"{uid}.npy")
            if uid in existing and (mkey != "cpsam_fp16" or os.path.exists(m_path)):
                var_rows.append(existing[uid])
                if mkey == "cpsam_fp16":
                    fp16_masks[uid] = np.load(m_path)
                continue

            rgb = load_rgb(r["image_path"])
            H, W = rgb.shape[:2]
            gt_masks = load_gt_mask(r["mask_path"], (H, W))
            gt_grains = compute_mask_grain_metrics(gt_masks)
            gt_b = area_weighted_percentiles(gt_grains, "bAxisPx")
            pt_masks = np.load(os.path.join(PT_CACHE_DIR, mkind, f"{uid}.npy")).astype(np.int32)
            pt_grains = compute_mask_grain_metrics(pt_masks)
            pt_b = area_weighted_percentiles(pt_grains, "bAxisPx")

            rgba = np.ascontiguousarray(np.dstack([rgb, np.full((H, W), 255, dtype=np.uint8)]), dtype=np.uint8)
            with tempfile.TemporaryDirectory() as td:
                rgba_p, dp_p, cp_p = os.path.join(td, "img.rgba"), os.path.join(td, "dp.f32"), os.path.join(td, "cp.f32")
                rgba.tofile(rgba_p)
                proc.stdin.write(json.dumps({"width": int(W), "height": int(H), "rgbaPath": rgba_p, "outDpPath": dp_p, "outCpPath": cp_p}) + "\n")
                proc.stdin.flush()
                resp = json.loads(proc.stdout.readline())
                dP = np.fromfile(dp_p, dtype="<f4").reshape((2, H, W))
                cp = np.fromfile(cp_p, dtype="<f4").reshape((H, W))

            ts_masks, ts_grains = run_ts_postprocess(dP, cp, 0.40)
            np.save(m_path, ts_masks)
            if mkey == "cpsam_fp16":
                fp16_masks[uid] = ts_masks
                s_fp16 = {"pq": 1.0, "ap50": 1.0}
            elif mkind == "cpsam" and uid in fp16_masks:
                s_fp16 = compute_seg_metrics(ts_masks, fp16_masks[uid])
            else:
                s_fp16 = {"pq": float("nan"), "ap50": float("nan")}

            ts_int = [g for g in ts_grains if not g.get("touchesBorder", g.get("isBorder", False))]
            ts_b = area_weighted_percentiles(ts_int, "bAxisPx")
            ts_d = area_weighted_percentiles(ts_int, "dEqPx")
            s_gt = compute_seg_metrics(ts_masks, gt_masks)
            s_pt = compute_seg_metrics(ts_masks, pt_masks)

            var_rows.append({
                "uid": uid, "site": r["site"], "is_k1_overlap": r["is_k1_overlap"],
                "model_key": mkey, "model_kind": mkind, "backend": backend_label,
                "infer_ms": resp["elapsedMs"],
                "n_gt": len(gt_grains), "n_pt": len(pt_grains), "n_pred": len(ts_int),
                "grain_count_ratio_gt": len(ts_int) / max(1, len(gt_grains)),
                "grain_count_ratio_pt": len(ts_int) / max(1, len(pt_grains)),
                "gt_b_D16": gt_b["D16"], "gt_b_D50": gt_b["D50"], "gt_b_D84": gt_b["D84"],
                "pt_b_D16": pt_b["D16"], "pt_b_D50": pt_b["D50"], "pt_b_D84": pt_b["D84"],
                "pred_b_D16": ts_b["D16"], "pred_b_D50": ts_b["D50"], "pred_b_D84": ts_b["D84"],
                "pred_d_D50": ts_d["D50"],
                "pq_vs_gt": s_gt["pq"], "sq_vs_gt": s_gt["sq"], "rq_vs_gt": s_gt["rq"], "ap50_vs_gt": s_gt["ap50"], "ap75_vs_gt": s_gt["ap75"],
                "pq_vs_pt": s_pt["pq"], "sq_vs_pt": s_pt["sq"], "rq_vs_pt": s_pt["rq"], "ap50_vs_pt": s_pt["ap50"], "ap75_vs_pt": s_pt["ap75"],
                "pq_vs_fp16": s_fp16["pq"], "ap50_vs_fp16": s_fp16["ap50"],
            })
            pd.DataFrame(var_rows).to_csv(var_csv, index=False)
            if DRIVE_MOUNTED:
                shutil.copy2(var_csv, drive_csv)
    finally:
        if proc is not None:
            try:
                proc.stdin.write(json.dumps({"quit": True}) + "\n")
                proc.stdin.flush()
                proc.wait(timeout=10)
            except Exception:
                proc.kill()
        if srv is not None:
            srv.shutdown()
    all_records.extend(var_rows)

df_all = pd.DataFrame(all_records)
per_img_csv = os.path.join(OUT_DIR, "final_4model_per_image.csv")
df_all.to_csv(per_img_csv, index=False)
if DRIVE_MOUNTED:
    shutil.copy2(per_img_csv, os.path.join(DRIVE_OUT_DIR, "final_4model_per_image.csv"))
print(f"Saved {per_img_csv} with {len(df_all)} rows across {df_all['model_key'].nunique()} models/references.")


In [ ]:
# Cell 5: Full statistical analysis, paired comparisons, per-site breakdown, outlier audit, & headline charts
import json, os, shutil
from math import comb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

def boot_ci(arr: np.ndarray, fn, n_boot: int = N_BOOT, seed: int = SEED) -> list[float]:
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return [float("nan"), float("nan")]
    if len(arr) == 1:
        v = float(fn(arr)); return [v, v]
    rng = np.random.default_rng(seed)
    boots = np.array([fn(arr[rng.integers(0, len(arr), size=len(arr))]) for _ in range(n_boot)], dtype=np.float64)
    return [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))]

def grain_error_stats(sub: pd.DataFrame, ref_prefix: str, pct: str) -> dict:
    rcol, pcol = f"{ref_prefix}_b_{pct}", f"pred_b_{pct}"
    valid = sub[sub[rcol].notna() & sub[pcol].notna() & (sub[rcol] > 0) & (sub[pcol] > 0)]
    if len(valid) == 0:
        return {"n": 0}
    ref = valid[rcol].to_numpy(dtype=np.float64)
    pred = valid[pcol].to_numpy(dtype=np.float64)
    err = pred - ref
    sre = 100.0 * err / ref
    ape = np.abs(sre)
    pr, _ = pearsonr(ref, pred) if len(ref) >= 3 else (float("nan"), float("nan"))
    sr, _ = spearmanr(ref, pred) if len(ref) >= 3 else (float("nan"), float("nan"))
    log_ref, log_pred = np.log10(ref), np.log10(pred)
    ss_res = float(np.sum((log_pred - log_ref) ** 2))
    ss_tot = float(np.sum((log_ref - np.mean(log_ref)) ** 2))
    r2_log = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return {
        "n": int(len(valid)),
        "mean_ape_pct": float(np.mean(ape)),
        "mean_ape_ci95": boot_ci(ape, np.mean),
        "median_ape_pct": float(np.median(ape)),
        "median_ape_ci95": boot_ci(ape, np.median),
        "p90_ape_pct": float(np.percentile(ape, 90)),
        "max_ape_pct": float(np.max(ape)),
        "mean_bias_pct": float(np.mean(sre)),
        "median_bias_pct": float(np.median(sre)),
        "mae_px": float(np.mean(np.abs(err))),
        "rmse_px": float(np.sqrt(np.mean(err ** 2))),
        "pearson_r": float(pr),
        "spearman_rho": float(sr),
        "r2_log10": float(r2_log),
    }

def summarize_model(sub: pd.DataFrame) -> dict:
    out = {"n_images": int(len(sub))}
    for metric in ("pq_vs_gt", "sq_vs_gt", "rq_vs_gt", "ap50_vs_gt", "ap75_vs_gt",
                   "pq_vs_pt", "sq_vs_pt", "rq_vs_pt", "ap50_vs_pt", "ap75_vs_pt",
                   "pq_vs_fp16", "ap50_vs_fp16",
                   "grain_count_ratio_gt", "grain_count_ratio_pt", "infer_ms"):
        arr = sub[metric].to_numpy(dtype=np.float64)
        arr = arr[np.isfinite(arr)]
        if len(arr):
            out[metric] = {
                "mean": float(np.mean(arr)),
                "ci95": boot_ci(arr, np.mean),
                "median": float(np.median(arr)),
                "min": float(np.min(arr)),
                "max": float(np.max(arr)),
            }
    out["catastrophic_gt_lt_020"] = int(((sub["pq_vs_gt"] < 0.20) | (sub["n_pred"] == 0)).sum())
    out["catastrophic_pt_lt_020"] = int(((sub["pq_vs_pt"] < 0.20) | (sub["n_pred"] == 0)).sum())
    out["vs_gt"] = {pct: grain_error_stats(sub, "gt", pct) for pct in ("D16", "D50", "D84")}
    out["vs_pt"] = {pct: grain_error_stats(sub, "pt", pct) for pct in ("D16", "D50", "D84")}
    return out

def paired_compare(df: pd.DataFrame, key_a: str, key_b: str, ref_prefix: str = "gt") -> dict:
    a = df[df["model_key"] == key_a].sort_values("uid")
    b = df[df["model_key"] == key_b].sort_values("uid")
    assert list(a["uid"]) == list(b["uid"]), f"UID mismatch: {key_a} vs {key_b}"
    ape_a = 100.0 * np.abs(a["pred_b_D50"].to_numpy() - a[f"{ref_prefix}_b_D50"].to_numpy()) / a[f"{ref_prefix}_b_D50"].to_numpy()
    ape_b = 100.0 * np.abs(b["pred_b_D50"].to_numpy() - b[f"{ref_prefix}_b_D50"].to_numpy()) / b[f"{ref_prefix}_b_D50"].to_numpy()
    d_ape = ape_a - ape_b
    d_pq = a["pq_vs_gt"].to_numpy() - b["pq_vs_gt"].to_numpy()
    rng = np.random.default_rng(0)
    boots_ape = np.array([rng.choice(d_ape, d_ape.size).mean() for _ in range(PAIRED_BOOT)])
    boots_pq = np.array([rng.choice(d_pq, d_pq.size).mean() for _ in range(PAIRED_BOOT)])
    lo_a, hi_a = np.percentile(boots_ape, [2.5, 97.5])
    lo_p, hi_p = np.percentile(boots_pq, [2.5, 97.5])
    wins = int((d_ape < 0).sum())
    losses = int((d_ape > 0).sum())
    n_nz = wins + losses
    k = min(wins, losses)
    p_val = min(1.0, 2.0 * sum(comb(n_nz, i) for i in range(k + 1)) / (2 ** n_nz)) if n_nz > 0 else 1.0
    verdict = "NOT distinguishable" if (lo_a <= 0 <= hi_a) else (f"{key_a} better" if hi_a < 0 else f"{key_b} better")
    return {
        "A": key_a, "B": key_b, "ref": ref_prefix, "n": int(len(d_ape)),
        "mean_dAPE_pp": float(np.mean(d_ape)), "dAPE_ci95": [float(lo_a), float(hi_a)],
        "A_wins": wins, "losses": losses, "ties": int(len(d_ape) - n_nz), "sign_test_p": float(p_val),
        "mean_dPQ_gt": float(np.mean(d_pq)), "dPQ_gt_ci95": [float(lo_p), float(hi_p)],
        "verdict_d50_ape": verdict,
    }

summary = {
    "hf_repo": HF_REPO, "hf_revision": HF_REVISION, "seed": SEED, "eval_images": len(test_rows),
    "models": {}, "pytorch_refs": {}, "paired_comparisons": [], "per_site": {}, "outliers": {},
}

for mkey in ("unet_fp32", "cpsam_fp16", "cpsam_int8", "cpsam_fp4"):
    sub = df_all[df_all["model_key"] == mkey]
    summary["models"][mkey] = {
        "meta": {k: v for k, v in model_meta[mkey].items() if k != "path"},
        "tiles": tile_results[mkey],
        "heldout_40": summarize_model(sub),
    }

for pkey in ("pt_unet", "pt_cpsam"):
    summary["pytorch_refs"][pkey] = summarize_model(df_all[df_all["model_key"] == pkey])

pairs = [
    ("unet_fp32", "pt_unet", "gt"),
    ("cpsam_fp16", "pt_cpsam", "gt"),
    ("cpsam_int8", "pt_cpsam", "gt"),
    ("cpsam_fp4", "pt_cpsam", "gt"),
    ("cpsam_int8", "cpsam_fp16", "gt"),
    ("cpsam_fp4", "cpsam_fp16", "gt"),
    ("cpsam_int8", "cpsam_fp4", "gt"),
    ("cpsam_int8", "cpsam_fp16", "pt"),
    ("cpsam_fp4", "cpsam_fp16", "pt"),
    ("cpsam_fp16", "unet_fp32", "gt"),
]
for ka, kb, rp in pairs:
    summary["paired_comparisons"].append(paired_compare(df_all, ka, kb, rp))

for site, grp in df_all.groupby("site"):
    summary["per_site"][site] = {}
    for mkey in ("pt_unet", "unet_fp32", "pt_cpsam", "cpsam_fp16", "cpsam_int8", "cpsam_fp4"):
        s = grp[grp["model_key"] == mkey]
        ape_gt = 100.0 * np.abs(s["pred_b_D50"] - s["gt_b_D50"]) / s["gt_b_D50"]
        ape_pt = 100.0 * np.abs(s["pred_b_D50"] - s["pt_b_D50"]) / s["pt_b_D50"]
        summary["per_site"][site][mkey] = {
            "n": int(len(s)),
            "pq_vs_gt": float(s["pq_vs_gt"].mean()),
            "pq_vs_pt": float(s["pq_vs_pt"].mean()),
            "d50_med_ape_gt": float(np.median(ape_gt)),
            "d50_med_ape_pt": float(np.median(ape_pt)),
        }

for mkey in ("unet_fp32", "cpsam_fp16", "cpsam_int8", "cpsam_fp4"):
    s = df_all[df_all["model_key"] == mkey].copy()
    s["d50_ape_gt"] = 100.0 * np.abs(s["pred_b_D50"] - s["gt_b_D50"]) / s["gt_b_D50"]
    s["d50_ape_pt"] = 100.0 * np.abs(s["pred_b_D50"] - s["pt_b_D50"]) / s["pt_b_D50"]
    worst_ape = s.sort_values("d50_ape_gt", ascending=False).head(5)[["uid", "site", "gt_b_D50", "pt_b_D50", "pred_b_D50", "d50_ape_gt", "d50_ape_pt", "pq_vs_gt", "pq_vs_pt"]]
    worst_pq = s.sort_values("pq_vs_gt", ascending=True).head(5)[["uid", "site", "pq_vs_gt", "pq_vs_pt", "d50_ape_gt"]]
    summary["outliers"][mkey] = {
        "worst_d50_ape_gt": worst_ape.to_dict(orient="records"),
        "lowest_pq_vs_gt": worst_pq.to_dict(orient="records"),
    }

res_json_p = os.path.join(OUT_DIR, "final_4model_results.json")
json.dump(summary, open(res_json_p, "w"), indent=2)
if DRIVE_MOUNTED:
    shutil.copy2(res_json_p, os.path.join(DRIVE_OUT_DIR, "final_4model_results.json"))

print("\n=== HEADLINE COMPARISON TABLE (40 Held-Out IG2 Test Images) ===\n")
hdr = "| Model / Ref | Raw (Brotli) MiB | Load (s) | WASM ms/tile | Peak RAM | All-Zero cp | PQ vs GT | F1@0.5 vs GT | D50 Med (Mean) APE vs GT | D50 Bias vs GT | PQ vs PT | D50 Med APE vs PT |"
print(hdr)
print("| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |")
for key, is_pt in [("pt_unet", True), ("unet_fp32", False), ("pt_cpsam", True), ("cpsam_fp16", False), ("cpsam_int8", False), ("cpsam_fp4", False)]:
    h = summary["pytorch_refs"][key] if is_pt else summary["models"][key]["heldout_40"]
    if is_pt:
        sz_str, ld_str, ms_str, ram_str, az_str = "PyTorch", "—", f"{h['infer_ms']['median']:.0f}/img", "—", "—"
    else:
        sz = summary["models"][key]["meta"]["size"]
        tw = summary["models"][key]["tiles"]["wasm"]
        tn = summary["models"][key]["tiles"]["numerics_vs_pt_fp32"]
        sz_str = f"{sz['mb']:.1f} ({sz['brotli_mb']:.1f})"
        ld_str = f"{tw['load_time_s']:.2f}"
        ms_str = f"{tw['avg_tile_ms']:.0f}"
        ram_str = f"{tw['peak_rss_mb']:.0f} MB"
        az_str = f"{tn['all_zero_cellprob_mean']:.2f}"
    d50g = h["vs_gt"]["D50"]
    d50p = h["vs_pt"]["D50"]
    print(f"| {key:10s} | {sz_str:16s} | {ld_str:8s} | {ms_str:12s} | {ram_str:8s} | {az_str:11s} | {h['pq_vs_gt']['mean']:.4f} | {h['rq_vs_gt']['mean']:.4f} | {d50g['median_ape_pct']:5.2f}% ({d50g['mean_ape_pct']:5.2f}%) | {d50g['median_bias_pct']:+5.2f}% | {h['pq_vs_pt']['mean']:.4f} | {d50p['median_ape_pct']:5.2f}% |")

print("\n=== PAIRED D50 APE & PQ COMPARISONS (10,000-draw bootstrap) ===")
for c in summary["paired_comparisons"]:
    print(f"A={c['A']:10s} vs B={c['B']:10s} ({c['ref'].upper()}) | dAPE={c['mean_dAPE_pp']:+5.2f} pp [{c['dAPE_ci95'][0]:+5.2f}, {c['dAPE_ci95'][1]:+5.2f}] | "
          f"wins={c['A_wins']}/{c['n']} p={c['sign_test_p']:.3f} | dPQ={c['mean_dPQ_gt']:+.4f} | {c['verdict_d50_ape']}")

plt.rcParams.update({"font.size": 10, "figure.dpi": 140})

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
keys4 = ["unet_fp32", "cpsam_fp16", "cpsam_int8", "cpsam_fp4"]
labels4 = [HF_MODELS[k]["label"] for k in keys4]
br_sizes = [summary["models"][k]["meta"]["size"]["brotli_mb"] for k in keys4]
pqs = [summary["models"][k]["heldout_40"]["pq_vs_gt"]["mean"] for k in keys4]
d50s = [summary["models"][k]["heldout_40"]["vs_gt"]["D50"]["median_ape_pct"] for k in keys4]
rams = [summary["models"][k]["tiles"]["wasm"]["peak_rss_mb"] / 1024.0 for k in keys4]
colors4 = ["#4c78a8", "#f58518", "#54a24b", "#e45756"]

for ax, ys, ylabel, title in [
    (axes[0], pqs, "Mean PQ vs Ground Truth", "Download Size vs Segmentation PQ"),
    (axes[1], d50s, "Median D50 APE vs GT (%)", "Download Size vs D50 Grain-Size Error"),
    (axes[2], rams, "Peak WASM RAM (GB)", "Download Size vs Browser Peak Memory"),
]:
    for x, y, lbl, col in zip(br_sizes, ys, labels4, colors4):
        ax.scatter(x, y, s=130, color=col, edgecolor="black", zorder=3)
        ax.annotate(lbl, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=9)
    ax.set_xlabel("Brotli Download Size (MiB)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
axes[0].axhline(summary["pytorch_refs"]["pt_cpsam"]["pq_vs_gt"]["mean"], ls="--", color="#888", label="PyTorch CP-SAM")
axes[0].legend(loc="lower right", fontsize=8)
plt.tight_layout()
p1 = os.path.join(OUT_DIR, "headline_tradeoff.png")
plt.savefig(p1); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for k, col in zip(keys4, colors4):
    s = df_all[df_all["model_key"] == k]
    axes[0].scatter(s["gt_b_D50"], s["pred_b_D50"], alpha=0.7, s=28, color=col, label=HF_MODELS[k]["label"])
    axes[1].scatter(s["pt_b_D50"], s["pred_b_D50"], alpha=0.7, s=28, color=col, label=HF_MODELS[k]["label"])
for ax, xlabel, title in [
    (axes[0], "Ground Truth D50 (px)", "Predicted vs Ground Truth D50"),
    (axes[1], "PyTorch Reference D50 (px)", "Shipped ONNX vs PyTorch D50"),
]:
    lims = [10, 400]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(xlabel); ax.set_ylabel("Predicted D50 (px)"); ax.set_title(title)
    ax.grid(True, which="both", alpha=0.25); ax.legend(fontsize=8)
plt.tight_layout()
p2 = os.path.join(OUT_DIR, "per_image_d50_parity.png")
plt.savefig(p2); plt.show()

sites = sorted(summary["per_site"].keys())
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(sites))
w = 0.14
for i, (k, col) in enumerate(zip(["pt_unet", "unet_fp32", "pt_cpsam", "cpsam_fp16", "cpsam_int8", "cpsam_fp4"],
                                 ["#9ecae9", "#4c78a8", "#ffbf79", "#f58518", "#54a24b", "#e45756"])):
    vals = [summary["per_site"][st][k]["pq_vs_gt"] for st in sites]
    ax.bar(x + (i - 2.5) * w, vals, width=w, label=k, color=col, edgecolor="black", linewidth=0.4)
ax.set_xticks(x); ax.set_xticklabels(sites)
ax.set_ylabel("Mean PQ vs GT"); ax.set_title("Segmentation Accuracy (PQ vs GT) by IG2 Site")
ax.set_ylim(0, 0.85); ax.grid(axis="y", alpha=0.3); ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
p3 = os.path.join(OUT_DIR, "site_pq_breakdown.png")
plt.savefig(p3); plt.show()

if DRIVE_MOUNTED:
    for p in (p1, p2, p3):
        shutil.copy2(p, os.path.join(DRIVE_OUT_DIR, "plots", os.path.basename(p)))
